# Small Dataset Comparison: ML vs DL on Tabular Data

**Dataset:** Breast Cancer Wisconsin (Diagnostic) - 569 rows x 30 numeric features, binary classification.

**What this notebook does:**
1. Loads the dataset and creates one fixed train / val / test split with stratification.
2. Fits one shared `StandardScaler` on the training fold and reuses it everywhere, so every model sees identical inputs.
3. Trains four models on the same scaled data:
   - **Logistic Regression** - linear, explainable ML baseline.
   - **XGBoost** - gradient-boosted decision trees, the strong tabular ML model the project requirement calls out.
   - **MLP (sklearn)** - a small feed-forward neural net, the simplest DL baseline.
   - **TabNet** - attention-based DL architecture *designed for tabular data*, the proper DL counterpart to XGBoost.
4. Reports accuracy, macro precision / recall / F1, ROC-AUC, training time, and inference time, plus confusion matrices and ROC curves.

**Why this comparison is fair:** every model is trained on the *same scaled training matrix* and evaluated on the *same scaled test matrix*. No model-specific feature engineering, no augmentation, no leakage.

In [ ]:
# Install the extra libraries used in this notebook (XGBoost + pytorch-tabnet).
# If they are already present this is a no-op.
try:
    import xgboost  # noqa: F401
except ImportError:
    %pip install -q xgboost
try:
    import pytorch_tabnet  # noqa: F401
except ImportError:
    %pip install -q pytorch-tabnet

In [ ]:
from pathlib import Path
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd()
DATA_CANDIDATES = [
    PROJECT_ROOT / "dataset" / "datasetSmall" / "cancer.csv",
    PROJECT_ROOT / "Data Set small" / "cancer.csv",
    PROJECT_ROOT / "Data set small" / "cancer.csv",
    PROJECT_ROOT / "datasetSmall" / "cancer.csv",
]


def resolve_existing_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find cancer.csv. Checked: " + ", ".join(str(p) for p in candidates)
    )


DATA_PATH = resolve_existing_path(DATA_CANDIDATES)
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "small_dataset"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
MODELS_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

for folder in [ARTIFACTS_DIR, FIGURES_DIR, MODELS_DIR, METRICS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Using dataset: {DATA_PATH}")
print(f"Artifacts folder: {ARTIFACTS_DIR}")
print(f"Torch device available: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1) Load the dataset

Read the CSV and split it into a feature matrix `X` and target vector `y`. The original CSV labels tumours as `M` (malignant) or `B` (benign); we encode them as `1` and `0` so every model can consume the same numeric target.

In [ ]:
df = pd.read_csv(DATA_PATH)

# Some local copies of this dataset have an extra trailing comma that creates an empty column.
empty_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
if empty_cols:
    df = df.drop(columns=empty_cols)

display(df.head())
print(f"Shape: {df.shape}")
print("\nDtypes:")
print(df.dtypes.value_counts())

TARGET_COL = "diagnosis"
ID_COL = "id"

if TARGET_COL not in df.columns:
    raise ValueError(f"Expected target column '{TARGET_COL}' not found in CSV.")

feature_cols = [c for c in df.columns if c not in [TARGET_COL, ID_COL]]
X_raw = df[feature_cols].copy()
y_raw = df[TARGET_COL].copy()

print(f"Feature count: {len(feature_cols)}")
print(f"Target column: {TARGET_COL}")

## 2) Validate the data and encode the target

We confirm there are no missing values or duplicates, then map `B -> 0` and `M -> 1`. We also check the class ratio so we know whether the problem is roughly balanced (it isn't perfectly - about 63% benign / 37% malignant, which is fine for stratified splitting).

In [ ]:
missing_values = df.isna().sum().sum()
duplicate_rows = df.duplicated().sum()

print(f"Total missing values: {missing_values}")
print(f"Duplicate rows: {duplicate_rows}")
print("\nTarget raw classes:")
print(y_raw.value_counts())

if y_raw.dtype == object:
    y = y_raw.map({"B": 0, "M": 1})
    if y.isna().any():
        unique_classes = sorted(y_raw.unique())
        class_to_int = {cls: idx for idx, cls in enumerate(unique_classes)}
        y = y_raw.map(class_to_int)
        print("Used fallback target mapping:", class_to_int)
    else:
        print("Used target mapping: {'B': 0, 'M': 1}")
else:
    y = y_raw.astype(int)
    print("Target already numeric.")

X = X_raw.apply(pd.to_numeric, errors="raise")

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print("Class balance (ratio):")
print(y.value_counts(normalize=True).sort_index())

## 3) Stratified train / validation / test split (70 / 15 / 15)

We create the split **once** and reuse it for every model. Stratification keeps the class ratio identical across the three folds, which matters when comparing models on a moderately imbalanced dataset.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE,
)

val_ratio_within_train_val = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=val_ratio_within_train_val,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

print("Split sizes:")
print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

print("\nClass balance by split:")
for split_name, y_split in {"train": y_train, "val": y_val, "test": y_test}.items():
    print(f"{split_name}:")
    print(y_split.value_counts(normalize=True).sort_index())
    print()

## 4) Fit the shared preprocessing pipeline

We fit **one** `StandardScaler` on the training fold only, then transform train / val / test with that same scaler. Every model in this notebook consumes these scaled arrays - `X_train_scaled`, `X_val_scaled`, `X_test_scaled` - so the comparison stays apples-to-apples.

Strictly speaking, tree-based models like XGBoost don't need scaling, but using the same input keeps the protocol identical and doesn't hurt performance.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

y_train_arr = y_train.to_numpy()
y_val_arr = y_val.to_numpy()
y_test_arr = y_test.to_numpy()

print("Scaled feature arrays ready:")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_val_scaled:   {X_val_scaled.shape}")
print(f"X_test_scaled:  {X_test_scaled.shape}")

## 5) Helper functions for scoring and metrics

`get_positive_class_scores` returns the predicted probability of the positive class - every model here exposes `predict_proba`, so we can extract that score uniformly.

`compute_common_metrics` packages accuracy, macro precision / recall / F1, and ROC-AUC into a dict. Using the **same function** for every model is what makes the final table directly comparable.

In [ ]:
def get_positive_class_scores(model, X_input):
    proba = model.predict_proba(X_input)
    if proba.ndim == 2 and proba.shape[1] > 1:
        return proba[:, 1]
    return proba.ravel()


def compute_common_metrics(y_true, y_pred, y_score):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_score)),
    }


def evaluate_on_val(model):
    val_pred = model.predict(X_val_scaled)
    val_score = get_positive_class_scores(model, X_val_scaled)
    return compute_common_metrics(y_val_arr, val_pred, val_score)

## 6a) Logistic Regression (linear ML baseline)

A simple, explainable model - the coefficients tell us which features push predictions toward malignant. It's the floor we expect the more powerful models to clear.

In [ ]:
baseline_models = {}
baseline_results = []

logreg_baseline = LogisticRegression(
    C=1.0, penalty="l2", solver="lbfgs",
    max_iter=2000, class_weight="balanced",
    random_state=RANDOM_STATE,
)

start_train = time.perf_counter()
logreg_baseline.fit(X_train_scaled, y_train_arr)
logreg_train_time = time.perf_counter() - start_train

val_metrics_lr = evaluate_on_val(logreg_baseline)
baseline_models["LogisticRegression"] = logreg_baseline
baseline_results.append({
    "model": "LogisticRegression",
    "train_time_sec": logreg_train_time,
    **{f"val_{k}": v for k, v in val_metrics_lr.items()},
})

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logreg_baseline.coef_.ravel(),
    "abs_coefficient": np.abs(logreg_baseline.coef_.ravel()),
}).sort_values("abs_coefficient", ascending=False)

print("Logistic Regression validation metrics:")
print(pd.Series(val_metrics_lr))
print("\nTop 10 features by |coefficient|:")
display(coef_df.head(10))

## 6b) XGBoost (gradient-boosted trees, the strong ML model)

XGBoost is the workhorse of tabular ML. It builds an ensemble of shallow decision trees where each new tree corrects the residual errors of the previous ones. It usually beats linear models comfortably on tabular data and is what the teacher specifically asked us to compare against deep learning.

In [ ]:
xgb_baseline = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    objective="binary:logistic", eval_metric="logloss",
    tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
)

start_train = time.perf_counter()
xgb_baseline.fit(X_train_scaled, y_train_arr)
xgb_train_time = time.perf_counter() - start_train

val_metrics_xgb = evaluate_on_val(xgb_baseline)
baseline_models["XGBoost"] = xgb_baseline
baseline_results.append({
    "model": "XGBoost",
    "train_time_sec": xgb_train_time,
    **{f"val_{k}": v for k, v in val_metrics_xgb.items()},
})

print("XGBoost validation metrics:")
print(pd.Series(val_metrics_xgb))

## 6c) MLP (small feed-forward neural network, simple DL baseline)

A two-hidden-layer perceptron from scikit-learn. It's the most basic 'deep learning on tabular' baseline - useful to show that simply throwing a generic neural net at tabular data isn't always a win.

In [ ]:
mlp_baseline = MLPClassifier(
    hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
    alpha=1e-4, learning_rate_init=1e-3, batch_size=32,
    max_iter=400, early_stopping=True, n_iter_no_change=20,
    random_state=RANDOM_STATE,
)

start_train = time.perf_counter()
mlp_baseline.fit(X_train_scaled, y_train_arr)
mlp_train_time = time.perf_counter() - start_train

val_metrics_mlp = evaluate_on_val(mlp_baseline)
baseline_models["MLPClassifier"] = mlp_baseline
baseline_results.append({
    "model": "MLPClassifier",
    "train_time_sec": mlp_train_time,
    **{f"val_{k}": v for k, v in val_metrics_mlp.items()},
})

print("MLP validation metrics:")
print(pd.Series(val_metrics_mlp))

## 6d) TabNet (attention-based DL specifically for tabular data)

TabNet (Arik & Pfister, 2019) is the deep-learning architecture **designed for tabular data**. It uses sequential attention to pick which features to look at at each decision step - conceptually similar to how a tree splits, but learned end-to-end with gradient descent. It's the proper DL counterpart to XGBoost.

We feed it the same scaled training matrix and use its built-in early stopping on the validation fold.

In [ ]:
tabnet_baseline = TabNetClassifier(
    n_d=16, n_a=16, n_steps=4, gamma=1.5, lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 20, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type="entmax", seed=RANDOM_STATE, verbose=0,
)

start_train = time.perf_counter()
tabnet_baseline.fit(
    X_train=X_train_scaled, y_train=y_train_arr,
    eval_set=[(X_val_scaled, y_val_arr)],
    eval_metric=["auc"],
    max_epochs=200, patience=20,
    batch_size=64, virtual_batch_size=32, drop_last=False,
)
tabnet_train_time = time.perf_counter() - start_train

val_metrics_tn = evaluate_on_val(tabnet_baseline)
baseline_models["TabNet"] = tabnet_baseline
baseline_results.append({
    "model": "TabNet",
    "train_time_sec": tabnet_train_time,
    **{f"val_{k}": v for k, v in val_metrics_tn.items()},
})

print("TabNet validation metrics:")
print(pd.Series(val_metrics_tn))

## 7) Validation summary across all four models

A single table comparing how each model did on the validation fold. This is a sanity check before we touch the test set.

In [ ]:
baseline_df = pd.DataFrame(baseline_results).set_index("model")
display(baseline_df.style.format("{:.4f}", subset=baseline_df.select_dtypes(float).columns))

## 8) Hyperparameter search on the validation fold

For LR and MLP we run a small `ParameterGrid`. For XGBoost and TabNet we keep the search compact. Selection is driven entirely by validation macro-F1, with ROC-AUC as a tie-breaker. The test set stays untouched until step 9.

In [ ]:
def is_better(candidate, incumbent):
    if incumbent is None:
        return True
    if candidate["val_f1_macro"] > incumbent["val_f1_macro"]:
        return True
    if (candidate["val_f1_macro"] == incumbent["val_f1_macro"]
            and candidate["val_roc_auc"] > incumbent["val_roc_auc"]):
        return True
    return False


def run_sklearn_search(model_name, base_estimator, param_grid):
    records, best_record = [], None
    for params in ParameterGrid(param_grid):
        model = clone(base_estimator).set_params(**params)
        start = time.perf_counter()
        model.fit(X_train_scaled, y_train_arr)
        train_time_sec = time.perf_counter() - start
        val_metrics = evaluate_on_val(model)
        record = {
            "model_family": model_name, "params": params,
            "train_time_sec": train_time_sec,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        records.append(record)
        if is_better(record, best_record):
            best_record = record
    return pd.DataFrame(records), best_record


logreg_grid = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "class_weight": [None, "balanced"],
    "solver": ["lbfgs"], "max_iter": [2000],
}
search_logreg_df, best_logreg_record = run_sklearn_search(
    "LogisticRegression", LogisticRegression(random_state=RANDOM_STATE), logreg_grid,
)

mlp_grid = {
    "hidden_layer_sizes": [(64, 32), (128, 64), (64, 64, 32)],
    "alpha": [1e-4, 1e-3],
    "learning_rate_init": [1e-3, 5e-4],
    "max_iter": [400],
}
search_mlp_df, best_mlp_record = run_sklearn_search(
    "MLPClassifier",
    MLPClassifier(random_state=RANDOM_STATE, early_stopping=True, n_iter_no_change=20),
    mlp_grid,
)

xgb_configs = [
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 400, "max_depth": 4, "learning_rate": 0.1},
    {"n_estimators": 600, "max_depth": 4, "learning_rate": 0.05},
    {"n_estimators": 600, "max_depth": 6, "learning_rate": 0.05},
]
xgb_records, best_xgb_record = [], None
for params in xgb_configs:
    full = {**params, "subsample": 0.9, "colsample_bytree": 0.9,
            "objective": "binary:logistic", "eval_metric": "logloss",
            "tree_method": "hist", "random_state": RANDOM_STATE, "n_jobs": -1}
    model = XGBClassifier(**full)
    start = time.perf_counter()
    model.fit(X_train_scaled, y_train_arr)
    train_time_sec = time.perf_counter() - start
    val_metrics = evaluate_on_val(model)
    record = {"model_family": "XGBoost", "params": params,
              "train_time_sec": train_time_sec,
              **{f"val_{k}": v for k, v in val_metrics.items()}}
    xgb_records.append(record)
    if is_better(record, best_xgb_record):
        best_xgb_record = record
search_xgb_df = pd.DataFrame(xgb_records)

tabnet_configs = [
    {"n_d": 8,  "n_a": 8,  "n_steps": 3, "gamma": 1.3, "lr": 2e-2},
    {"n_d": 16, "n_a": 16, "n_steps": 4, "gamma": 1.5, "lr": 2e-2},
    {"n_d": 24, "n_a": 24, "n_steps": 5, "gamma": 1.5, "lr": 1e-2},
]
tabnet_records, best_tabnet_record = [], None
for params in tabnet_configs:
    model = TabNetClassifier(
        n_d=params["n_d"], n_a=params["n_a"], n_steps=params["n_steps"],
        gamma=params["gamma"], lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=params["lr"]),
        scheduler_params={"step_size": 20, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type="entmax", seed=RANDOM_STATE, verbose=0,
    )
    start = time.perf_counter()
    model.fit(
        X_train=X_train_scaled, y_train=y_train_arr,
        eval_set=[(X_val_scaled, y_val_arr)],
        eval_metric=["auc"], max_epochs=200, patience=20,
        batch_size=64, virtual_batch_size=32, drop_last=False,
    )
    train_time_sec = time.perf_counter() - start
    val_metrics = evaluate_on_val(model)
    record = {"model_family": "TabNet", "params": params,
              "train_time_sec": train_time_sec,
              **{f"val_{k}": v for k, v in val_metrics.items()}}
    tabnet_records.append(record)
    if is_better(record, best_tabnet_record):
        best_tabnet_record = record
search_tabnet_df = pd.DataFrame(tabnet_records)

search_results = pd.concat(
    [search_logreg_df, search_mlp_df, search_xgb_df, search_tabnet_df],
    ignore_index=True,
)

display(
    search_results[[
        "model_family", "val_accuracy", "val_precision_macro", "val_recall_macro",
        "val_f1_macro", "val_roc_auc", "train_time_sec", "params"
    ]].sort_values(["model_family", "val_f1_macro", "val_roc_auc"], ascending=[True, False, False])
)

print("Best per family:")
for rec in [best_logreg_record, best_xgb_record, best_mlp_record, best_tabnet_record]:
    print(f"  {rec['model_family']}: F1={rec['val_f1_macro']:.4f} AUC={rec['val_roc_auc']:.4f} params={rec['params']}")

## 9) Refit each winner and evaluate on the held-out test set

We retrain each best configuration once more (so the saved artifact reflects the chosen config) and then score on the test fold. The test set has not been touched since the very first split, which is what makes these numbers an honest estimate of generalisation.

In [ ]:
def build_logreg(params):
    return LogisticRegression(random_state=RANDOM_STATE).set_params(**params)


def build_mlp(params):
    return MLPClassifier(
        random_state=RANDOM_STATE, early_stopping=True, n_iter_no_change=20,
    ).set_params(**params)


def build_xgb(params):
    full = {**params, "subsample": 0.9, "colsample_bytree": 0.9,
            "objective": "binary:logistic", "eval_metric": "logloss",
            "tree_method": "hist", "random_state": RANDOM_STATE, "n_jobs": -1}
    return XGBClassifier(**full)


def build_tabnet(params):
    return TabNetClassifier(
        n_d=params["n_d"], n_a=params["n_a"], n_steps=params["n_steps"],
        gamma=params["gamma"], lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=params["lr"]),
        scheduler_params={"step_size": 20, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type="entmax", seed=RANDOM_STATE, verbose=0,
    )


def fit_tabnet(model):
    model.fit(
        X_train=X_train_scaled, y_train=y_train_arr,
        eval_set=[(X_val_scaled, y_val_arr)],
        eval_metric=["auc"], max_epochs=200, patience=20,
        batch_size=64, virtual_batch_size=32, drop_last=False,
    )


selection_plan = {
    "LogisticRegression": (build_logreg, best_logreg_record["params"], "sklearn"),
    "XGBoost":            (build_xgb,    best_xgb_record["params"],    "sklearn"),
    "MLPClassifier":      (build_mlp,    best_mlp_record["params"],    "sklearn"),
    "TabNet":             (build_tabnet, best_tabnet_record["params"], "tabnet"),
}

selected_models = {}
selected_model_info = {}
for model_family, (builder, params, kind) in selection_plan.items():
    model = builder(params)
    start = time.perf_counter()
    if kind == "tabnet":
        fit_tabnet(model)
    else:
        model.fit(X_train_scaled, y_train_arr)
    train_time_sec = time.perf_counter() - start
    val_metrics = evaluate_on_val(model)
    selected_models[model_family] = model
    selected_model_info[model_family] = {
        "best_params": params, "train_time_sec": train_time_sec,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

display(pd.DataFrame(selected_model_info).T)

In [ ]:
test_results = {}
model_predictions = {}
model_scores = {}
for model_family, model in selected_models.items():
    y_pred = model.predict(X_test_scaled)
    y_score = get_positive_class_scores(model, X_test_scaled)
    test_results[model_family] = compute_common_metrics(y_test_arr, y_pred, y_score)
    model_predictions[model_family] = y_pred
    model_scores[model_family] = y_score

test_results_df = pd.DataFrame(test_results).T
display(test_results_df.style.format("{:.4f}"))

## 10) Confusion matrices and ROC curves

Visualising the test-set predictions side by side. The confusion matrices show *where* each model makes mistakes (false positives vs false negatives matter very differently in a cancer screening context), and the ROC curves show how well each model separates the two classes across all thresholds.

In [ ]:
label_names = ["Benign (0)", "Malignant (1)"]

n_models = len(selected_models)
fig_cm, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4.5))
if n_models == 1:
    axes = [axes]

for ax, (model_family, y_pred) in zip(axes, model_predictions.items()):
    cm = confusion_matrix(y_test_arr, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(f"Confusion Matrix - {model_family}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

fig_roc, ax_roc = plt.subplots(figsize=(7, 6))
for model_family, y_score in model_scores.items():
    fpr, tpr, _ = roc_curve(y_test_arr, y_score)
    auc_value = roc_auc_score(y_test_arr, y_score)
    ax_roc.plot(fpr, tpr, label=f"{model_family} (AUC={auc_value:.4f})")

ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
ax_roc.set_title("ROC Curve on Test Set")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(loc="lower right")
ax_roc.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 11) Training and inference time

Accuracy isn't the only thing that matters. We benchmark how long each model took to train and how fast it can score one sample at inference time. On a small dataset like this you'll typically see Logistic Regression and XGBoost training in milliseconds, while MLP and especially TabNet take noticeably longer.

In [ ]:
timing_rows = []
n_repeats = 100

for model_family, model in selected_models.items():
    train_time_sec = selected_model_info[model_family]["train_time_sec"]
    start_infer = time.perf_counter()
    for _ in range(n_repeats):
        _ = model.predict(X_test_scaled)
    infer_total_sec = time.perf_counter() - start_infer
    infer_per_sample_ms = (infer_total_sec / (n_repeats * len(X_test_scaled))) * 1000
    timing_rows.append({
        "model_family": model_family,
        "train_time_sec": train_time_sec,
        "inference_total_sec": infer_total_sec,
        "inference_per_sample_ms": infer_per_sample_ms,
    })

timing_df = pd.DataFrame(timing_rows).set_index("model_family")
display(timing_df.style.format("{:.6f}"))

## 12) Final comparison table and saved artifacts

Everything in one place: validation metrics, test metrics, training time, inference time, and the chosen hyperparameters per model. We also persist the trained models, the scaler, and all the metric / search CSVs under `artifacts/small_dataset/` so they can be referenced without rerunning the notebook.

In [ ]:
final_rows = []
for model_family in selected_models.keys():
    info = selected_model_info[model_family]
    final_rows.append({
        "model_family": model_family,
        "val_accuracy": info["val_accuracy"],
        "val_precision_macro": info["val_precision_macro"],
        "val_recall_macro": info["val_recall_macro"],
        "val_f1_macro": info["val_f1_macro"],
        "val_roc_auc": info["val_roc_auc"],
        "test_accuracy": test_results[model_family]["accuracy"],
        "test_precision_macro": test_results[model_family]["precision_macro"],
        "test_recall_macro": test_results[model_family]["recall_macro"],
        "test_f1_macro": test_results[model_family]["f1_macro"],
        "test_roc_auc": test_results[model_family]["roc_auc"],
        "train_time_sec": timing_df.loc[model_family, "train_time_sec"],
        "inference_per_sample_ms": timing_df.loc[model_family, "inference_per_sample_ms"],
        "best_params": json.dumps(info["best_params"]),
    })

final_comparison_df = pd.DataFrame(final_rows).sort_values("test_f1_macro", ascending=False)
display(final_comparison_df)

comparison_csv_path = METRICS_DIR / "final_comparison_small_dataset.csv"
comparison_json_path = METRICS_DIR / "final_comparison_small_dataset.json"
search_csv_path = METRICS_DIR / "validation_search_results_small_dataset.csv"
coefficients_csv_path = METRICS_DIR / "logreg_coefficients_small_dataset.csv"

search_results_to_save = search_results.copy()
search_results_to_save["params"] = search_results_to_save["params"].apply(json.dumps)
search_results_to_save.to_csv(search_csv_path, index=False)
final_comparison_df.to_csv(comparison_csv_path, index=False)

with open(comparison_json_path, "w", encoding="utf-8") as f:
    json.dump(final_rows, f, indent=2)

coef_df.to_csv(coefficients_csv_path, index=False)

for model_family, model in selected_models.items():
    if model_family == "TabNet":
        model.save_model(str(MODELS_DIR / "TabNet_best"))
    else:
        joblib.dump(model, MODELS_DIR / f"{model_family}_best.joblib")

joblib.dump(scaler, MODELS_DIR / "standard_scaler.joblib")

with open(METRICS_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

cm_plot_path = FIGURES_DIR / "confusion_matrices_test.png"
roc_plot_path = FIGURES_DIR / "roc_curves_test.png"
fig_cm.savefig(cm_plot_path, dpi=200, bbox_inches="tight")
fig_roc.savefig(roc_plot_path, dpi=200, bbox_inches="tight")

print("Saved artifacts:")
for p in [comparison_csv_path, comparison_json_path, search_csv_path,
         coefficients_csv_path, cm_plot_path, roc_plot_path,
         MODELS_DIR / "standard_scaler.joblib"]:
    print(f"- {p}")

## 13) Takeaways for the class

- **Same data, different models.** Every result above came from one shared train / val / test split and one shared scaler. Differences in performance are differences in the models, not in the data.
- **Logistic Regression is hard to beat on small, well-behaved data.** With only 569 rows and 30 informative features, a linear model already nails this problem.
- **XGBoost adds non-linear interactions cheaply** and usually edges ahead, with very little training cost.
- **MLP and TabNet are competitive but expensive.** They train slower, and on a dataset this small they don't have enough rows to clearly outpace the simpler models. This is the headline finding of the small-dataset side of the study.
- **The story may flip on bigger data.** That's exactly what the Covertype notebook is for.